# ML Lab 4
Name: Anushka Chikte
Roll No: 53
Section:C
Batch:C4

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('winequality-red.csv')
df.head()

In [ ]:
df.tail()

In [ ]:
df.shape

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
# Missing values and duplicate analysis
print(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())

df = df.drop_duplicates()
print("Duplicates after removal:", df.duplicated().sum())

## Univariate Analysis

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns

for col in numeric_cols:
    plt.figure(figsize=(7,4))
    sns.histplot(df[col], kde=True)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.show()

In [ ]:
# Boxplot – Outlier Analysis
for col in numeric_cols:
    plt.figure(figsize=(7,4))
    sns.boxplot(x=df[col])
    plt.title(f"Boxplot of {col}")
    plt.show()

In [ ]:
# Quality Distribution
plt.figure(figsize=(7,4))
sns.countplot(x='quality', data=df)
plt.title('Distribution of Wine Quality')
plt.show()

## Bivariate Analysis

In [ ]:
features = ['fixed acidity','volatile acidity','citric acid','residual sugar',
            'chlorides','free sulfur dioxide','total sulfur dioxide','density',
            'pH','sulphates','alcohol']

for col in features:
    plt.figure(figsize=(7,4))
    sns.boxplot(x='quality', y=col, data=df)
    plt.title(f'{col} vs Wine Quality')
    plt.show()

In [ ]:
sns.scatterplot(x='alcohol', y='quality', data=df)
plt.title('Alcohol vs Quality')
plt.show()

In [ ]:
sns.scatterplot(x='volatile acidity', y='quality', data=df)
plt.title('Volatile Acidity vs Quality')
plt.show()

## Multivariate Analysis

In [ ]:
plt.figure(figsize=(9,7))
corr = df.corr(numeric_only=True)
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
selected = ['alcohol','volatile acidity','sulphates','citric acid','quality']
sns.pairplot(df[selected], hue='quality')
plt.show()

In [ ]:
df['quality_group'] = pd.cut(
    df['quality'],
    bins=[0,5,6,10],
    labels=['Low','Medium','High']
)

sns.boxplot(x='quality_group', y='alcohol', data=df)
plt.title('Alcohol Distribution Across Quality Groups')
plt.show()

## Decision Tree Classification

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, auc, roc_auc_score
)
from sklearn.preprocessing import label_binarize

X = df.drop(columns=['quality', 'quality_group'], errors='ignore')
y = df['quality']

print("X shape", X.shape)
print("y shape", y.shape)
print(y.value_counts().sort_index())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print("X_train", X_train.shape)
print("X_test", X_test.shape)
print("y_train", y_train.shape)
print("y_test", y_test.shape)

In [ ]:
dt_model = DecisionTreeClassifier(
    criterion='entropy',
    random_state=42
)
dt_model.fit(X_train, y_train)

In [ ]:
y_train_pred = dt_model.predict(X_train)
y_test_pred = dt_model.predict(X_test)

train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print("Training Accuracy:", train_accuracy)
print("Testing Accuracy:", test_accuracy)
print("Precision:", precision_score(y_test, y_test_pred, average='weighted', zero_division=0))
print("Recall:", recall_score(y_test, y_test_pred, average='weighted', zero_division=0))
print("F1 Score:", f1_score(y_test, y_test_pred, average='weighted', zero_division=0))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_test_pred)
print("Confusion Matrix:")
print(cm)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
from sklearn.tree import plot_tree

plt.figure(figsize=(24,12))
plot_tree(
    dt_model,
    feature_names=X.columns,
    class_names=[str(c) for c in sorted(y.unique())],
    filled=True,
    rounded=True,
    fontsize=8
)
plt.title('Decision Tree Visualization')
plt.show()

print("Tree Depth:", dt_model.get_depth())
print("Number of leaves:", dt_model.get_n_leaves())

## AUC ROC Curve

In [ ]:
classes = sorted(y_test.unique())
y_test_bin = label_binarize(y_test, classes=classes)
y_score = dt_model.predict_proba(X_test)

plt.figure(figsize=(10,7))

for i, class_label in enumerate(classes):
    fpr, tpr, thresholds = roc_curve(
        y_test_bin[:, i],
        y_score[:, i]
    )
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, linewidth=2,
             label=f'Quality {class_label} (AUC={roc_auc:.3f})')

plt.plot([0,1], [0,1], linestyle='--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve-Before Hyperparameter Tuning')
plt.legend()
plt.grid()
plt.show()

overall_auc_before = roc_auc_score(
    y_test_bin, y_score, multi_class='ovr', average='macro'
)
print("Overall ROC-AUC Before Tuning:", round(overall_auc_before, 4))